# 1. Reference CNN

Three convolution/ReLU/pooling blocks (32, 64, 128 channels), adaptive average pooling, dropout 0.3 and eight raw logits. There are 94,280 trainable parameters. Adam uses learning rate 0.001, with at most 15 epochs.

Inputs: saved split, mapping, original configuration and training images. The saved run is displayed below. A new run writes exclusively to a separate reproduction directory. No training starts by default.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

# Set this explicitly on Kaggle to the extracted project folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "src/models.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the extracted project containing src/ and data/processed/")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.project import RUNS, load_metadata, read_strings, find_image_directory, find_csv
DATA_ROOT = (Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2")
             if Path("/kaggle").exists() else PROJECT_ROOT / "data/raw")
OUTPUT_ROOT = (Path("/kaggle/working/reproduction") if Path("/kaggle").exists()
               else PROJECT_ROOT / "outputs/reproduction")
TRAIN_IMAGE_DIRECTORY = None  # Optional existing cache or explicit train/train directory.
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUN_TRAINING = False
RUN_TEST_PREDICTION = False
RUN_EVALUATION = False
split, mapping = load_metadata(PROJECT_ROOT)
print("Project:", PROJECT_ROOT, "Device:", DEVICE)
print(split["split"].value_counts().to_dict())

In [ ]:
from src.models import build_model
from src.experiment import train_experiment, plot_history
EXPERIMENT = "E0"
RUN_DIRECTORY = OUTPUT_ROOT / "E0_reproduction_seed42"
original = PROJECT_ROOT / "outputs/metrics" / RUNS[EXPERIMENT]
configuration = json.loads((original / "configuration.json").read_text())
history = pd.read_csv(original / "history.csv")
display(history)
print("Original best epoch:", int(history.loc[history["validation_loss"].idxmin(), "epoch"]))
model = build_model(EXPERIMENT)  # Architecture only: no pretrained download.
print("Parameters:", sum(p.numel() for p in model.parameters()),
      "Trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Reference CNN with three convolutional blocks and eight raw class scores.")
del model

## 2. Training policy

Use the unchanged 50,927/12,733 split, seed 42, batch 32, workers 0 and float32. Shuffle training with a seeded generator; keep validation in saved order. Use unweighted cross-entropy on raw logits and sample-weighted losses/accuracies, including final partial batches. Save the lowest-validation-loss checkpoint. ReduceLROnPlateau uses a patience of 2 and halves the learning rate when a plateau persists. Stop after three consecutive epochs without a lower validation loss.

A new directory is required; existing outputs are never resumed or overwritten. Reproduction is seeded, but GPU and cross-version bitwise agreement is not promised. CPU execution is supported for development; the reported experiments used Kaggle GPU execution.

In [ ]:
reproduced_history = None  # Reset so a failed or skipped run cannot show stale results.
if RUN_TRAINING:
    directory = TRAIN_IMAGE_DIRECTORY or find_image_directory(DATA_ROOT, split["image_id"])
    reproduced_history = train_experiment(PROJECT_ROOT, directory, EXPERIMENT, RUN_DIRECTORY, DEVICE)
else:
    print("Training disabled; showing the preserved experiment history only.")

## 3. Learning curves

After a successful new run, display that run's history and learning curves. Otherwise display the preserved original history. The label identifies which experiment is shown. Original histories, figures, configurations and checkpoints remain unchanged.

In [ ]:
import matplotlib.pyplot as plt
if RUN_TRAINING and reproduced_history is not None:
    displayed_history = reproduced_history
    display_label = f"{EXPERIMENT}: new reproduction run ({RUN_DIRECTORY.name})"
else:
    displayed_history = history
    display_label = f"{EXPERIMENT}: preserved original run ({original.name})"
print(display_label)
display(displayed_history)
figure = plot_history(displayed_history)
figure.suptitle(display_label, fontsize=11)
plt.show()